In [1]:
import pandas as pd

roll_number = 1024170053

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

last_two_digits = [int(digit) for digit in str(roll_number)[-2:]]

categories = ["billing", "account", "general"]

personalized_entries = [
    {
        "question": "how do i change my registered email",
        "answer": "You can change your registered email from account settings.",
        "keywords": "email change settings",
        "category": categories[last_two_digits[0] % 3]
    },
    {
        "question": "how can i check my recent payment",
        "answer": "You can check your recent payment in the billing section.",
        "keywords": "payment history billing",
        "category": categories[last_two_digits[1] % 3]
    }
]

faq_df = pd.DataFrame(fixed_entries + personalized_entries)

print(faq_df)

                              question  \
0               what is the annual fee   
1                how to reset password   
2          what are your working hours   
3                how can i pay the fee   
4  how do i change my registered email   
5    how can i check my recent payment   

                                              answer                 keywords  \
0                          The annual fee is Rs 500.    fee cost price charge   
1                   Go to Settings > Reset Password.     password reset login   
2                          We are open 9 AM to 5 PM.   hours timing open time   
3         You can pay via UPI, card, or net banking.      pay payment upi fee   
4  You can change your registered email from acco...    email change settings   
5  You can check your recent payment in the billi...  payment history billing   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  general  
5  billing  


In [2]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())
        score = len(query_words & keywords)

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    results.sort(key=lambda x: x["confidence"], reverse=True)

    return pd.DataFrame(results)


query = input("Enter your query: ")

result = score_query(query, faq_df)

print(result)

Enter your query: fee
   index                question                                      answer  \
0      0  what is the annual fee                   The annual fee is Rs 500.   
1      3   how can i pay the fee  You can pay via UPI, card, or net banking.   

  category  confidence  
0  billing           1  
1  billing           1  


In [3]:
def same_category(category_name, df):
    return df[df["category"] == category_name]

category_name = faq_df.iloc[4]["category"]

result = same_category(category_name, faq_df)

print(result)

                              question  \
2          what are your working hours   
4  how do i change my registered email   

                                              answer                keywords  \
2                          We are open 9 AM to 5 PM.  hours timing open time   
4  You can change your registered email from acco...   email change settings   

  category  
2  general  
4  general  


In [4]:
entry_index = 0

new_keyword = input("Enter a new keyword: ")

faq_df.loc[entry_index, "keywords"] = faq_df.loc[entry_index, "keywords"] + " " + new_keyword

faq_df.to_csv("1024170053_faq_data.csv", index=False)

print(faq_df)

Enter a new keyword: fee
                              question  \
0               what is the annual fee   
1                how to reset password   
2          what are your working hours   
3                how can i pay the fee   
4  how do i change my registered email   
5    how can i check my recent payment   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  You can change your registered email from acco...   
5  You can check your recent payment in the billi...   

                    keywords category  
0  fee cost price charge fee  billing  
1       password reset login  account  
2     hours timing open time  general  
3        pay payment upi fee  billing  
4      email change settings  general  
5    payment history billing  billing  


In [5]:
category_counts = faq_df.groupby("category").size()

print(category_counts)

category
account    1
billing    3
general    2
dtype: int64


In [6]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())
        score = len(query_words & keywords)

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    if not results:
        print("No matching entries found")
        return

    highest_score = max(result["confidence"] for result in results)

    highest_matches = [
        result for result in results
        if result["confidence"] == highest_score
    ]

    if len(highest_matches) > 1:
        print("Multiple entries have the highest score:")
        print(pd.DataFrame(highest_matches))
    else:
        print("Best matching entry:")
        print(pd.DataFrame(highest_matches))

    results.sort(key=lambda x: x["confidence"], reverse=True)

    print("All ranked matches:")
    print(pd.DataFrame(results))

In [7]:
score_query_with_ties("fee", faq_df)

Multiple entries have the highest score:
   index                question                                      answer  \
0      0  what is the annual fee                   The annual fee is Rs 500.   
1      3   how can i pay the fee  You can pay via UPI, card, or net banking.   

  category  confidence  
0  billing           1  
1  billing           1  
All ranked matches:
   index                question                                      answer  \
0      0  what is the annual fee                   The annual fee is Rs 500.   
1      3   how can i pay the fee  You can pay via UPI, card, or net banking.   

  category  confidence  
0  billing           1  
1  billing           1  


In [8]:
score_query_with_ties("password", faq_df)

Best matching entry:
   index               question                            answer category  \
0      1  how to reset password  Go to Settings > Reset Password.  account   

   confidence  
0           1  
All ranked matches:
   index               question                            answer category  \
0      1  how to reset password  Go to Settings > Reset Password.  account   

   confidence  
0           1  
